# LeetCode 739 — Daily Temperatures bằng Python

Link đề: https://leetcode.com/problems/daily-temperatures/ · Độ khó: **Medium** · Chủ đề: Array, Stack, Monotonic Stack

Skeleton LeetCode:

```python
class Solution:
    def dailyTemperatures(self, temperatures: List[int]) -> List[int]:
        ...
```

Ghi chú: đây là bài kinh điển để học **ngăn xếp đơn điệu** (monotonic stack): với mỗi phần tử, tìm phần tử **đầu tiên lớn hơn nó** nằm bên phải. Notebook đi từ cách quét tìm trực tiếp `O(n^2)`, qua ngăn xếp đơn điệu `O(n)`, qua cách "nhảy theo đáp án đã có" từ phải sang trái `O(n)` bộ nhớ phụ `O(1)`, và cuối cùng là cách tận dụng ràng buộc nhiệt độ chỉ trong `30..100`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: với mỗi ngày `i`, cần biết sau **bao nhiêu ngày** nữa thì gặp ngày **ấm hơn hẳn** (nhiệt độ lớn hơn); không có thì ghi `0`.
2. Nắm các công cụ Python cần dùng: danh sách làm **ngăn xếp** (`append`, `pop`, `[-1]`), `enumerate`, duyệt ngược `range(n - 1, -1, -1)`, vòng `while stack and điều_kiện`, và nhân danh sách `[0] * n`.
3. Đi từ **quét tìm từng ngày (`O(n^2)`)** → **ngăn xếp đơn điệu chứa chỉ số** → **nhảy theo đáp án đã tính từ phải sang trái** → **bảng "chỉ số gần nhất của mỗi nhiệt độ"** (khai thác `30 <= t <= 100`).
4. Hiểu mấu chốt: ngăn xếp luôn chứa các ngày **chưa tìm được ngày ấm hơn**, và nhiệt độ của chúng **không tăng** từ đáy lên đỉnh; khi một ngày mới ấm hơn đỉnh thì **giải quyết** (pop) đỉnh, rồi tiếp tục với phần tử dưới.
5. Tránh các lỗi kinh điển: dùng `<=` thay vì `<` khi pop (ngày bằng nhiệt độ không phải "ấm hơn"), lưu nhiệt độ thay vì **chỉ số** vào ngăn xếp, khởi tạo đáp án khác `0`, và quét tuyến tính tưởng là đủ nhanh.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `temperatures` là nhiệt độ mỗi ngày. Trả về mảng `answer` sao cho `answer[i]` là **số ngày phải chờ** sau ngày `i` để gặp một ngày có nhiệt độ **cao hơn**. Nếu không có ngày nào như vậy thì `answer[i] = 0`.

**Ví dụ 1**

- Input: `temperatures = [73,74,75,71,69,72,76,73]`
- Output: `[1,1,4,2,1,1,0,0]`

**Ví dụ 2**

- Input: `temperatures = [30,40,50,60]`
- Output: `[1,1,1,0]`

**Ví dụ 3**

- Input: `temperatures = [30,60,90]`
- Output: `[1,1,0]`

**Ràng buộc:**

- `1 <= temperatures.length <= 10^5`
- `30 <= temperatures[i] <= 100`

Ví dụ 1 giải thích: ngày 2 (nhiệt độ `75`) phải chờ tới ngày 6 (nhiệt độ `76`), tức `6 - 2 = 4` ngày. "Ấm hơn" nghĩa là **lớn hơn hẳn**: cùng nhiệt độ không tính.

In [1]:
from typing import List
from collections import deque
import random
import time

examples = [
    ([73, 74, 75, 71, 69, 72, 76, 73], [1, 1, 4, 2, 1, 1, 0, 0]),
    ([30, 40, 50, 60], [1, 1, 1, 0]),
    ([30, 60, 90], [1, 1, 0]),
]
for t, expected in examples:
    print(t, "->", expected)

[73, 74, 75, 71, 69, 72, 76, 73] -> [1, 1, 4, 2, 1, 1, 0, 0]
[30, 40, 50, 60] -> [1, 1, 1, 0]
[30, 60, 90] -> [1, 1, 0]


## 3. Kiến thức nền

### 3.1. Ngăn xếp (stack) bằng danh sách

Ngăn xếp là cấu trúc **vào sau ra trước** (LIFO). Với `list` của Python:

- `st.append(x)`: đẩy `x` lên đỉnh, `O(1)`.
- `st.pop()`: lấy ra và trả về phần tử ở đỉnh, `O(1)`.
- `st[-1]`: xem đỉnh mà không lấy ra.
- `while st:` / `if st:`: ngăn xếp không rỗng (danh sách rỗng được coi là `False`).

In [2]:
st = []
st.append(10); st.append(20); st.append(30)
print(st, "đỉnh =", st[-1])
print("pop ->", st.pop(), "còn", st)
print("rỗng?", not st, "| sau khi pop hết:", [st.pop(), st.pop()], "rỗng?", not st)

[10, 20, 30] đỉnh = 30
pop -> 30 còn [10, 20]
rỗng? False | sau khi pop hết: [20, 10] rỗng? True


### 3.2. `enumerate`, `[0] * n` và duyệt ngược

- `for i, x in enumerate(t)` cho đồng thời chỉ số `i` và giá trị `x`.
- `[0] * n` tạo danh sách `n` số `0` (đúng cho số, **không** dùng để tạo danh sách lồng nhau).
- `range(n - 1, -1, -1)` duyệt `i` từ `n - 1` về `0` (duyệt ngược).

In [3]:
t = [73, 74, 75]
for i, x in enumerate(t):
    print("ngày", i, "nhiệt độ", x)
print([0] * 4)
print(list(range(len(t) - 1, -1, -1)))

ngày 0 nhiệt độ 73
ngày 1 nhiệt độ 74
ngày 2 nhiệt độ 75
[0, 0, 0, 0]
[2, 1, 0]


### 3.3. Vì sao lưu **chỉ số** chứ không lưu giá trị

Đáp án cần **khoảng cách giữa hai ngày** `j - i`. Nếu chỉ lưu nhiệt độ vào ngăn xếp thì ta mất vị trí ngày; còn nếu lưu **chỉ số** thì vẫn tra được nhiệt độ qua `t[chỉ_số]`. Vì vậy ngăn xếp sẽ chứa chỉ số.

In [4]:
t = [73, 74, 75, 71]
st = [0, 2, 3]                       # chỉ số của các ngày
print([t[j] for j in st], "<- nhiệt độ tương ứng, tra qua chỉ số")
print("khoảng cách từ ngày 3 tới ngày 5 là", 5 - 3)

[73, 75, 71] <- nhiệt độ tương ứng, tra qua chỉ số
khoảng cách từ ngày 3 tới ngày 5 là 2


### 3.4. Ngăn xếp **đơn điệu** là gì

Ngăn xếp đơn điệu giữ một **tính chất sắp xếp** từ đáy lên đỉnh (ở đây: nhiệt độ **không tăng**). Mỗi khi thêm phần tử mới, ta **pop** các phần tử ở đỉnh vi phạm tính chất, rồi mới push. Mỗi phần tử được push một lần và pop tối đa một lần nên tổng thời gian là `O(n)` dù có vòng `while` lồng trong vòng `for` (**phân tích khấu hao**, amortized).

In [5]:
# Minh họa: giữ stack nhiệt độ không tăng; đếm số lần pop
t = [73, 74, 75, 71, 69, 72, 76, 73]
st, pops = [], 0
for x in t:
    while st and st[-1] < x:
        st.pop(); pops += 1
    st.append(x)
    print("thêm", x, "->", st)
print("tổng pop =", pops, "<= n =", len(t))

thêm 73 -> [73]
thêm 74 -> [74]
thêm 75 -> [75]
thêm 71 -> [75, 71]
thêm 69 -> [75, 71, 69]
thêm 72 -> [75, 72]
thêm 76 -> [76]
thêm 73 -> [76, 73]
tổng pop = 6 <= n = 8


### 3.5. Chi phí của `x in list` và khoảng nhiệt độ nhỏ

Đề cho nhiệt độ chỉ trong `30..100` (71 giá trị). Một mảng cỡ `101` chỉ tốn vài trăm byte, nên ta có thể **đánh chỉ mục theo nhiệt độ** (`arr[nhiệt_độ]`). Đó là ý của Cách 4.

In [6]:
nxt = [None] * 101
nxt[75] = 2
nxt[73] = 7
print(nxt[75], nxt[73], nxt[74])
print(min(j for j in (nxt[v] for v in range(73, 101)) if j is not None))

2 7 None
2


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Lấy ví dụ 1: `[73,74,75,71,69,72,76,73]`. Làm tay từng ngày bằng cách nhìn sang phải:

| Ngày `i` | `t[i]` | Nhìn sang phải tìm ngày đầu tiên ấm hơn | `answer[i]` |
|---|---|---|---|
| 0 | 73 | ngày 1 (74) | 1 |
| 1 | 74 | ngày 2 (75) | 1 |
| 2 | 75 | ngày 3..5 là 71, 69, 72 (không); ngày 6 là 76 | 4 |
| 3 | 71 | ngày 4 là 69 (không), ngày 5 là 72 | 2 |
| 4 | 69 | ngày 5 (72) | 1 |
| 5 | 72 | ngày 6 (76) | 1 |
| 6 | 76 | không còn ngày nào ấm hơn | 0 |
| 7 | 73 | hết mảng | 0 |

Nhận xét: khi tìm cho ngày 2 (`75`), ta nhìn qua `71, 69, 72`; những ngày đó **chẳng bao giờ ấm hơn 75**, và rồi cũng chính các ngày đó ta lại phải đi qua khi xử lý ngày 3 và 4. Có sự **lặp lại công sức**.

Ý tưởng tiết kiệm: đừng "mỗi ngày tự đi tìm", hãy **duyệt từ trái sang phải một lần** và để **ngày mới đến "trả lời" cho các ngày đang chờ**. Các ngày đang chờ là những ngày chưa gặp ngày ấm hơn; ngày mới có nhiệt độ `x` sẽ trả lời cho **mọi ngày đang chờ có nhiệt độ nhỏ hơn `x`**.

## 5. Cách 1 — Brute Force: với mỗi ngày quét sang phải

### Ý tưởng

Đúng như bảng tính tay: với mỗi `i`, duyệt `j = i + 1, i + 2, ...` cho tới khi gặp `t[j] > t[i]`, khi đó `answer[i] = j - i`; nếu hết mảng mà không gặp thì giữ `0`.

### Code

In [7]:
def daily_brute(t):
    n = len(t)
    ans = [0] * n
    for i in range(n):
        for j in range(i + 1, n):
            if t[j] > t[i]:
                ans[i] = j - i
                break
    return ans

for t, expected in examples:
    print(t, "->", daily_brute(t), "(kỳ vọng", expected, ")")

[73, 74, 75, 71, 69, 72, 76, 73] -> [1, 1, 4, 2, 1, 1, 0, 0] (kỳ vọng [1, 1, 4, 2, 1, 1, 0, 0] )
[30, 40, 50, 60] -> [1, 1, 1, 0] (kỳ vọng [1, 1, 1, 0] )
[30, 60, 90] -> [1, 1, 0] (kỳ vọng [1, 1, 0] )


### Độ phức tạp và nút thắt

- Time: tệ nhất `O(n^2)`: ví dụ khi mọi nhiệt độ bằng nhau (hoặc giảm dần) thì mỗi ngày phải quét hết phần bên phải. Với `n = 10^5` là khoảng `5 · 10^9` phép so sánh.
- Space: `O(1)` phụ (ngoài mảng đáp án).

**Nút thắt:** các lần quét của những ngày khác nhau **đi qua cùng một đoạn** mà không tận dụng nhau. Hãy xem thời gian tăng như thế nào với mảng toàn nhiệt độ bằng nhau.

In [8]:
for n in [500, 1000, 2000, 4000]:
    t = [50] * n
    t0 = time.perf_counter()
    daily_brute(t)
    print(f"n={n}: {time.perf_counter() - t0:.4f}s")

n=500: 0.0052s
n=1000: 0.0253s
n=2000: 0.0971s


n=4000: 0.3155s


Thời gian tăng gấp khoảng 4 lần mỗi khi `n` gấp đôi: đúng `O(n^2)`. Với `n = 10^5` sẽ mất vài phút (ước lượng ở mục 13), không đạt giới hạn thời gian.

## 6. Ý tưởng thuật toán / bước đột phá

Gọi một ngày là **"đang chờ"** nếu nó chưa tìm được ngày ấm hơn. Duyệt `i` từ `0` tới `n - 1`, duy trì ngăn xếp `st` chứa **chỉ số** các ngày đang chờ.

> Khi tới ngày `i` với nhiệt độ `t[i]`: chừng nào đỉnh ngăn xếp `j` có `t[j] < t[i]` thì ngày `i` chính là ngày ấm hơn đầu tiên của `j`: ghi `answer[j] = i - j` và pop `j`. Sau đó push `i` (ngày `i` bắt đầu chờ).

Vì sao chỉ cần nhìn **đỉnh**? Ta luôn giữ nhiệt độ trong ngăn xếp **không tăng từ đáy lên đỉnh** (đỉnh nhỏ nhất). Nếu đỉnh `j` có `t[j] >= t[i]` thì mọi phần tử dưới nó có nhiệt độ `>= t[j] >= t[i]` nên cũng chưa được ngày `i` trả lời: có thể dừng.

Mỗi chỉ số push đúng một lần và pop tối đa một lần: tổng `O(n)`.

In [9]:
t = [73, 74, 75, 71, 69, 72, 76, 73]
ans = [0] * len(t)
st = []
for i, x in enumerate(t):
    popped = []
    while st and t[st[-1]] < x:
        j = st.pop()
        ans[j] = i - j
        popped.append(j)
    st.append(i)
    print(f"ngày {i} (t={x}): pop {popped} -> stack (chỉ số) = {st}, nhiệt độ = {[t[k] for k in st]}")
print(ans)

ngày 0 (t=73): pop [] -> stack (chỉ số) = [0], nhiệt độ = [73]
ngày 1 (t=74): pop [0] -> stack (chỉ số) = [1], nhiệt độ = [74]
ngày 2 (t=75): pop [1] -> stack (chỉ số) = [2], nhiệt độ = [75]
ngày 3 (t=71): pop [] -> stack (chỉ số) = [2, 3], nhiệt độ = [75, 71]
ngày 4 (t=69): pop [] -> stack (chỉ số) = [2, 3, 4], nhiệt độ = [75, 71, 69]
ngày 5 (t=72): pop [4, 3] -> stack (chỉ số) = [2, 5], nhiệt độ = [75, 72]
ngày 6 (t=76): pop [5, 2] -> stack (chỉ số) = [6], nhiệt độ = [76]
ngày 7 (t=73): pop [] -> stack (chỉ số) = [6, 7], nhiệt độ = [76, 73]
[1, 1, 4, 2, 1, 1, 0, 0]


## 7. Các cách tiếp theo

Ta có 4 cách khác nhau thực sự:

| Cách | Ý tưởng |
|---|---|
| 1 | Với mỗi ngày quét sang phải (mục 5) |
| 2 | **Ngăn xếp đơn điệu** chứa chỉ số các ngày đang chờ (cách nộp) |
| 3 | Duyệt **từ phải sang trái**, **nhảy** theo đáp án đã tính của các ngày bên phải |
| 4 | Duyệt từ phải sang trái, bảng `nxt[nhiệt_độ]` = chỉ số gần nhất của nhiệt độ đó; tra các nhiệt độ lớn hơn |

### Cách 2 — Ngăn xếp đơn điệu

**Giải thích từng đoạn:**

- `ans = [0] * n`: mặc định `0` nghĩa là "không có ngày ấm hơn"; các ngày còn lại trong ngăn xếp khi kết thúc sẽ giữ `0`.
- `for i, x in enumerate(t)`: xử lý từng ngày theo thứ tự.
- `while st and t[st[-1]] < x`: điều kiện `<` **chặt**: bằng nhau thì **không** pop vì "ấm hơn" là lớn hơn hẳn.
- `ans[j] = i - j`: ngày `i` là ngày ấm hơn đầu tiên của `j`.
- `st.append(i)`: ngày `i` bắt đầu chờ.

### Code demo

In [10]:
def daily_stack(t):
    n = len(t)
    ans = [0] * n
    st = []
    for i, x in enumerate(t):
        while st and t[st[-1]] < x:
            j = st.pop()
            ans[j] = i - j
        st.append(i)
    return ans

for t, expected in examples:
    print(t, "->", daily_stack(t), "(kỳ vọng", expected, ")")

[73, 74, 75, 71, 69, 72, 76, 73] -> [1, 1, 4, 2, 1, 1, 0, 0] (kỳ vọng [1, 1, 4, 2, 1, 1, 0, 0] )
[30, 40, 50, 60] -> [1, 1, 1, 0] (kỳ vọng [1, 1, 1, 0] )
[30, 60, 90] -> [1, 1, 0] (kỳ vọng [1, 1, 0] )


### Code nộp LeetCode (Cách 2)

```python
class Solution:
    def dailyTemperatures(self, temperatures: List[int]) -> List[int]:
        n = len(temperatures)
        ans = [0] * n
        st = []
        for i, x in enumerate(temperatures):
            while st and temperatures[st[-1]] < x:
                j = st.pop()
                ans[j] = i - j
            st.append(i)
        return ans
```

### Độ phức tạp

- Time: `O(n)` (mỗi chỉ số push/pop một lần).
- Space: `O(n)` cho ngăn xếp (tệ nhất khi nhiệt độ không tăng: mọi chỉ số nằm trong ngăn xếp).

### Cách 3 — Từ phải sang trái, nhảy theo đáp án đã có

**Ý tưởng khác biệt:** đi từ ngày cuối về ngày đầu. Với ngày `i`, ta bắt đầu ở `j = i + 1`. Nếu `t[j] > t[i]` thì xong: `answer[i] = j - i`. Nếu không, ta **không cần thử `j + 1`**: ngày `j` đã biết ngày ấm hơn đầu tiên của nó cách `answer[j]` ngày; mà `t[j] <= t[i]` nên các ngày giữa `j` và `j + answer[j]` đều không ấm hơn `j`, tức cũng không ấm hơn `i` (vì `t[i] >= t[j]`). Ta **nhảy** `j += answer[j]`. Nếu `answer[j] == 0` nghĩa là sau `j` không còn ngày nào ấm hơn `j`, cũng không ấm hơn `i`: dừng với `answer[i] = 0`.

### Code demo

In [11]:
def daily_jump(t):
    n = len(t)
    ans = [0] * n
    for i in range(n - 2, -1, -1):
        j = i + 1
        while t[j] <= t[i]:
            if ans[j] == 0:          # sau j không có ngày nào ấm hơn t[j] >= ... => cũng không ấm hơn t[i]
                j = n
                break
            j += ans[j]              # nhảy tới ngày ấm hơn đầu tiên của j
        if j < n:
            ans[i] = j - i
    return ans

for t, expected in examples:
    print(t, "->", daily_jump(t), "(kỳ vọng", expected, ")")

[73, 74, 75, 71, 69, 72, 76, 73] -> [1, 1, 4, 2, 1, 1, 0, 0] (kỳ vọng [1, 1, 4, 2, 1, 1, 0, 0] )
[30, 40, 50, 60] -> [1, 1, 1, 0] (kỳ vọng [1, 1, 1, 0] )
[30, 60, 90] -> [1, 1, 0] (kỳ vọng [1, 1, 0] )


### Code nộp LeetCode (Cách 3)

```python
class Solution:
    def dailyTemperatures(self, temperatures: List[int]) -> List[int]:
        t = temperatures
        n = len(t)
        ans = [0] * n
        for i in range(n - 2, -1, -1):
            j = i + 1
            while t[j] <= t[i]:
                if ans[j] == 0:
                    j = n
                    break
                j += ans[j]
            if j < n:
                ans[i] = j - i
        return ans
```

Chú ý: vòng `while` luôn có `j < n` vì ta chỉ ghi `j = n` rồi `break`; và nhảy `j += ans[j]` chỉ xảy ra khi `ans[j] > 0` nên `j` vẫn nằm trong mảng.

### Độ phức tạp

- Time: `O(n)` khấu hao (mỗi lần nhảy bỏ qua một đoạn đã biết; có thể chứng minh tổng số bước là `O(n)`).
- Space: `O(1)` phụ (ngoài mảng đáp án), không cần ngăn xếp.

### Cách 4 — Bảng `nxt[nhiệt_độ]` (khai thác `30 <= t <= 100`)

**Ý tưởng khác biệt:** đi từ phải sang trái, giữ `nxt[v]` = chỉ số **gần nhất bên phải** của ngày có nhiệt độ **đúng bằng** `v`. Với ngày `i`: ngày ấm hơn đầu tiên là ngày có chỉ số **nhỏ nhất** trong các `nxt[v]` với `v` từ `t[i] + 1` tới `100`. Tìm min trên tối đa `70` giá trị rồi cập nhật `nxt[t[i]] = i`.

### Code demo

In [12]:
def daily_by_value(t):
    n = len(t)
    ans = [0] * n
    nxt = [None] * 101                      # nxt[v] = chỉ số gần nhất bên phải có nhiệt độ v
    for i in range(n - 1, -1, -1):
        best = None
        for v in range(t[i] + 1, 101):      # chỉ xét các nhiệt độ lớn hơn hẳn
            j = nxt[v]
            if j is not None and (best is None or j < best):
                best = j
        if best is not None:
            ans[i] = best - i
        nxt[t[i]] = i
    return ans

for t, expected in examples:
    print(t, "->", daily_by_value(t), "(kỳ vọng", expected, ")")

[73, 74, 75, 71, 69, 72, 76, 73] -> [1, 1, 4, 2, 1, 1, 0, 0] (kỳ vọng [1, 1, 4, 2, 1, 1, 0, 0] )
[30, 40, 50, 60] -> [1, 1, 1, 0] (kỳ vọng [1, 1, 1, 0] )
[30, 60, 90] -> [1, 1, 0] (kỳ vọng [1, 1, 0] )


### Code nộp LeetCode (Cách 4)

```python
class Solution:
    def dailyTemperatures(self, temperatures: List[int]) -> List[int]:
        n = len(temperatures)
        ans = [0] * n
        nxt = [None] * 101
        for i in range(n - 1, -1, -1):
            best = None
            for v in range(temperatures[i] + 1, 101):
                j = nxt[v]
                if j is not None and (best is None or j < best):
                    best = j
            if best is not None:
                ans[i] = best - i
            nxt[temperatures[i]] = i
        return ans
```

### Độ phức tạp

- Time: `O(n · V)` với `V = 71` là số nhiệt độ khác nhau; vì `V` là hằng số theo đề nên coi như `O(n)`, nhưng hằng số lớn hơn Cách 2 và 3.
- Space: `O(V) = O(1)` phụ.

Cách này **phụ thuộc ràng buộc** `30 <= t <= 100`; nếu nhiệt độ có khoảng rất lớn thì không dùng được (khi đó dùng Cách 2).

## 8. TRACE từng bước

Hàm `trace_stack(t)` in từng ngày: giá trị, các lần pop (kèm đáp án vừa ghi), và trạng thái ngăn xếp.

Bảng trạng thái cho **ví dụ 1** `[73,74,75,71,69,72,76,73]`:

| Ngày `i` | `t[i]` | Pop (chỉ số `j` -> `i - j`) | Ngăn xếp sau bước (chỉ số) |
|---|---|---|---|
| 0 | 73 | không | `[0]` |
| 1 | 74 | `0 -> 1` | `[1]` |
| 2 | 75 | `1 -> 1` | `[2]` |
| 3 | 71 | không | `[2, 3]` |
| 4 | 69 | không | `[2, 3, 4]` |
| 5 | 72 | `4 -> 1`, `3 -> 2` | `[2, 5]` |
| 6 | 76 | `5 -> 1`, `2 -> 4` | `[6]` |
| 7 | 73 | không | `[6, 7]` |

Cuối cùng ngăn xếp còn `[6, 7]`: hai ngày này không có ngày ấm hơn nên giữ `0`. Kết quả `[1,1,4,2,1,1,0,0]`.

In [13]:
def trace_stack(t):
    n = len(t)
    ans = [0] * n
    st = []
    for i, x in enumerate(t):
        print(f"--- ngày {i}, t={x}")
        while st and t[st[-1]] < x:
            j = st.pop()
            ans[j] = i - j
            print(f"    t[{j}]={t[j]} < {x} -> pop {j}, answer[{j}] = {i} - {j} = {ans[j]}")
        if st:
            print(f"    đỉnh t[{st[-1]}]={t[st[-1]]} >= {x} -> dừng pop")
        st.append(i)
        print(f"    push {i} -> stack = {st} (nhiệt độ {[t[k] for k in st]})")
    print("còn trong stack (giữ 0):", st)
    return ans

print(trace_stack([73, 74, 75, 71, 69, 72, 76, 73]))

--- ngày 0, t=73
    push 0 -> stack = [0] (nhiệt độ [73])
--- ngày 1, t=74
    t[0]=73 < 74 -> pop 0, answer[0] = 1 - 0 = 1
    push 1 -> stack = [1] (nhiệt độ [74])
--- ngày 2, t=75
    t[1]=74 < 75 -> pop 1, answer[1] = 2 - 1 = 1
    push 2 -> stack = [2] (nhiệt độ [75])
--- ngày 3, t=71
    đỉnh t[2]=75 >= 71 -> dừng pop
    push 3 -> stack = [2, 3] (nhiệt độ [75, 71])
--- ngày 4, t=69
    đỉnh t[3]=71 >= 69 -> dừng pop
    push 4 -> stack = [2, 3, 4] (nhiệt độ [75, 71, 69])
--- ngày 5, t=72
    t[4]=69 < 72 -> pop 4, answer[4] = 5 - 4 = 1
    t[3]=71 < 72 -> pop 3, answer[3] = 5 - 3 = 2
    đỉnh t[2]=75 >= 72 -> dừng pop
    push 5 -> stack = [2, 5] (nhiệt độ [75, 72])
--- ngày 6, t=76
    t[5]=72 < 76 -> pop 5, answer[5] = 6 - 5 = 1
    t[2]=75 < 76 -> pop 2, answer[2] = 6 - 2 = 4
    push 6 -> stack = [6] (nhiệt độ [76])
--- ngày 7, t=73
    đỉnh t[6]=76 >= 73 -> dừng pop
    push 7 -> stack = [6, 7] (nhiệt độ [76, 73])
còn trong stack (giữ 0): [6, 7]
[1, 1, 4, 2, 1, 1, 0, 0]


In [14]:
# Trace nhiệt độ bằng nhau: không pop vì "ấm hơn" là lớn hơn hẳn
print(trace_stack([60, 60, 70]))

--- ngày 0, t=60
    push 0 -> stack = [0] (nhiệt độ [60])
--- ngày 1, t=60
    đỉnh t[0]=60 >= 60 -> dừng pop
    push 1 -> stack = [0, 1] (nhiệt độ [60, 60])
--- ngày 2, t=70
    t[1]=60 < 70 -> pop 1, answer[1] = 2 - 1 = 1
    t[0]=60 < 70 -> pop 0, answer[0] = 2 - 0 = 2
    push 2 -> stack = [2] (nhiệt độ [70])
còn trong stack (giữ 0): [2]
[2, 1, 0]


Ở trace thứ hai: ngày 1 có `60` bằng `60` của ngày 0 nên **không pop**; ngày 2 (`70`) pop cả hai: ngày 1 nhận `1`, ngày 0 nhận `2`. Nếu dùng `<=` thì ngày 0 sẽ nhận nhầm `1` (từ ngày 1 cùng nhiệt độ).

## 9. Code nộp LeetCode

Ta nộp Cách 2 (ngăn xếp đơn điệu): chuẩn mực, `O(n)`, dễ tổng quát hóa.

In [15]:
class Solution:
    def dailyTemperatures(self, temperatures: List[int]) -> List[int]:
        n = len(temperatures)
        ans = [0] * n
        st = []
        for i, x in enumerate(temperatures):
            while st and temperatures[st[-1]] < x:
                j = st.pop()
                ans[j] = i - j
            st.append(i)
        return ans


for t, expected in examples:
    got = Solution().dailyTemperatures(t)
    print(t, "->", got, "| kỳ vọng:", expected, "| OK" if got == expected else "| SAI")

[73, 74, 75, 71, 69, 72, 76, 73] -> [1, 1, 4, 2, 1, 1, 0, 0] | kỳ vọng: [1, 1, 4, 2, 1, 1, 0, 0] | OK
[30, 40, 50, 60] -> [1, 1, 1, 0] | kỳ vọng: [1, 1, 1, 0] | OK
[30, 60, 90] -> [1, 1, 0] | kỳ vọng: [1, 1, 0] | OK


## 10. Vì sao đúng

**Bất biến (invariant).** Sau khi xử lý ngày `i`, ngăn xếp chứa **đúng** các ngày `j <= i` chưa có ngày ấm hơn trong khoảng `(j, i]`, và nhiệt độ của chúng **không tăng** từ đáy lên đỉnh.

**Chứng minh bằng quy nạp.**

- Cơ sở: trước ngày `0` ngăn xếp rỗng, thỏa.
- Bước: giả sử bất biến đúng sau ngày `i - 1`. Khi tới ngày `i` với nhiệt độ `x`:
  - Một ngày `j` trong ngăn xếp **được trả lời bởi `i`** khi và chỉ khi `t[j] < x` (vì đó là các ngày chưa gặp ngày ấm hơn trước `i`, nên `i` là ngày **đầu tiên** ấm hơn nó).
  - Vì nhiệt độ không tăng từ đáy lên đỉnh, các ngày có `t[j] < x` tạo thành một **đoạn liên tiếp ở đỉnh**. Vòng `while` pop đúng các phần tử đó và dừng ở phần tử đầu tiên có `t >= x`, dưới nó mọi phần tử đều có `t >= x` nên không ngày nào khác được trả lời.
  - Sau khi pop, đỉnh có `t >= x`; push `i` giữ tính không tăng. Ngày `i` chưa có ngày ấm hơn nên đúng là ngày đang chờ.
- Kết thúc: các ngày còn trong ngăn xếp không có ngày ấm hơn ở bên phải, nên giữ `0`.

**Đáp án đúng khoảng cách.** `ans[j] = i - j` với `i` là ngày ấm hơn đầu tiên.

Dưới đây ta kiểm chứng bất biến bằng thí nghiệm: ở mọi bước, ngăn xếp không tăng, và mỗi ngày trong ngăn xếp thật sự chưa có ngày ấm hơn.

In [16]:
rng = random.Random(739)
steps = 0
for _ in range(500):
    n = rng.randint(1, 40)
    t = [rng.randint(30, 45) for _ in range(n)]      # khoảng hẹp để có nhiều nhiệt độ bằng nhau
    st = []
    for i, x in enumerate(t):
        while st and t[st[-1]] < x:
            st.pop()
        st.append(i)
        temps = [t[k] for k in st]
        assert all(temps[k] >= temps[k + 1] for k in range(len(temps) - 1))     # không tăng
        for j in st:
            assert all(t[m] <= t[j] for m in range(j + 1, i + 1))                # chưa có ngày ấm hơn
        steps += 1
print("bất biến đúng ở", steps, "bước")

bất biến đúng ở 10581 bước


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1 Quét sang phải | `O(n^2)` | `O(1)` |
| 2 Ngăn xếp đơn điệu | `O(n)` | `O(n)` |
| 3 Nhảy theo đáp án | `O(n)` khấu hao | `O(1)` |
| 4 Bảng theo nhiệt độ | `O(n · V)`, `V = 71` | `O(V)` |

Ta đo số lần `pop` và so thời gian thực của Cách 2, 3, 4 trên `n = 10^5`.

In [17]:
rng = random.Random(1)
n = 100000
inputs = {
    "ngẫu nhiên": [rng.randint(30, 100) for _ in range(n)],
    "tăng dần (lặp)": [30 + (i * 71) // n for i in range(n)],
    "không tăng": [100 - (i * 71) // n for i in range(n)],
    "toàn bằng nhau": [50] * n,
}
for name, t in inputs.items():
    res = []
    for fn in (daily_stack, daily_jump, daily_by_value):
        t0 = time.perf_counter()
        r = fn(t)
        res.append((r, round(time.perf_counter() - t0, 3)))
    assert res[0][0] == res[1][0] == res[2][0]
    print(f"{name:16}: stack {res[0][1]}s | nhảy {res[1][1]}s | theo nhiệt độ {res[2][1]}s")

# đếm số lần pop trong Cách 2: luôn <= n
def count_pops(t):
    st, pops = [], 0
    for i, x in enumerate(t):
        while st and t[st[-1]] < x:
            st.pop(); pops += 1
        st.append(i)
    return pops
print("pop nhiều nhất trong 4 input:", max(count_pops(t) for t in inputs.values()), "<= n =", n)

ngẫu nhiên      : stack 0.013s | nhảy 0.016s | theo nhiệt độ 0.149s


tăng dần (lặp)  : stack 0.013s | nhảy 0.017s | theo nhiệt độ 0.18s
không tăng      : stack 0.012s | nhảy 0.01s | theo nhiệt độ 0.091s


toàn bằng nhau  : stack 0.009s | nhảy 0.009s | theo nhiệt độ 0.127s
pop nhiều nhất trong 4 input: 98592 <= n = 100000


## 12. Bộ test

Các nhóm test: ví dụ của đề, edge case (một phần tử, toàn bằng nhau, tăng chặt, giảm, hai giá trị lặp), phản ví dụ (nhiệt độ bằng nhau — nơi `<=` làm sai), và đối chiếu **cả 4 cách** trên input ngẫu nhiên có seed đúng ràng buộc.

In [18]:
# 1) Ví dụ của đề
for t, expected in examples:
    assert Solution().dailyTemperatures(t) == expected

# 2) Edge case
assert Solution().dailyTemperatures([30]) == [0]
assert Solution().dailyTemperatures([100]) == [0]
assert Solution().dailyTemperatures([50, 50, 50]) == [0, 0, 0]              # bằng nhau: không ấm hơn
assert Solution().dailyTemperatures([30, 31, 32, 33]) == [1, 1, 1, 0]       # tăng chặt
assert Solution().dailyTemperatures([33, 32, 31, 30]) == [0, 0, 0, 0]       # giảm chặt
assert Solution().dailyTemperatures([60, 60, 70]) == [2, 1, 0]              # phản ví dụ cho '<='
assert Solution().dailyTemperatures([70, 60, 60, 80]) == [3, 2, 1, 0]
assert Solution().dailyTemperatures([89, 62, 70, 58, 47, 47, 46, 76, 100, 70]) == [8, 1, 5, 4, 3, 2, 1, 1, 0, 0]
assert Solution().dailyTemperatures([30, 100, 30, 100, 30]) == [1, 0, 1, 0, 0]
print("ví dụ + edge OK")

ví dụ + edge OK


In [19]:
# 3) Đối chiếu 5 cách trên input ngẫu nhiên có seed, đúng ràng buộc
rng = random.Random(739)
solvers = [daily_brute, daily_stack, daily_jump, daily_by_value, lambda t: Solution().dailyTemperatures(t)]
total = 0
for _ in range(2000):
    n = rng.randint(1, 60)
    lo = rng.randint(30, 99)
    hi = rng.randint(lo, min(100, lo + rng.choice([1, 3, 10, 70])))
    t = [rng.randint(lo, hi) for _ in range(n)]            # nhiều nhiệt độ bằng nhau khi khoảng hẹp
    expected = daily_brute(t)
    for fn in solvers:
        assert fn(t) == expected, (fn, t)
    total += 1
print("2000 input ngẫu nhiên x", len(solvers), "cách: OK")

# 4) Input dài hơn: bỏ Cách 1 cho tới độ dài 3000, còn lại dùng oracle độc lập: duyệt từ trái, deque các ngày chờ
def oracle(t):
    # oracle độc lập: duy trì danh sách ngày chờ (không đơn điệu), mỗi ngày mới quét toàn bộ danh sách
    waiting, ans = [], [0] * len(t)
    for i, x in enumerate(t):
        still = []
        for j in waiting:
            if t[j] < x:
                ans[j] = i - j
            else:
                still.append(j)
        waiting = still
        waiting.append(i)
    return ans

for _ in range(30):
    n = rng.randint(500, 3000)
    t = [rng.randint(30, 100) for _ in range(n)]
    e = oracle(t)
    assert Solution().dailyTemperatures(t) == e == daily_jump(t) == daily_by_value(t) == daily_brute(t)
print("input dài + oracle độc lập OK")

# 5) Giới hạn tối đa n = 10^5
rng = random.Random(100000)
t = [rng.randint(30, 100) for _ in range(100000)]
assert Solution().dailyTemperatures(t) == daily_jump(t) == daily_by_value(t)
big_equal = [77] * 100000
assert Solution().dailyTemperatures(big_equal) == [0] * 100000
print("n = 10^5 OK")

# 6) Đầu vào không bị sửa
t = [73, 74, 75, 71, 69, 72, 76, 73]
copy = t[:]
Solution().dailyTemperatures(t)
assert t == copy
print("đầu vào không bị sửa")

2000 input ngẫu nhiên x 5 cách: OK


input dài + oracle độc lập OK


n = 10^5 OK
đầu vào không bị sửa


## 13. Các lỗi người mới hay gặp

Ta cài vài hàm sai cố ý rồi tìm input nhỏ nhất làm chúng sai bằng `find_mismatch`.

In [20]:
def find_mismatch(fn, trials=3000, seed=7):
    rng = random.Random(seed)
    best = None
    for _ in range(trials):
        n = rng.randint(1, 7)
        t = [rng.randint(30, 34) for _ in range(n)]
        try:
            ok = fn(t) == daily_brute(t)
        except Exception:
            ok = False
        if not ok and (best is None or len(t) < len(best)):
            best = t
    return best

### Lỗi 1: dùng `<=` thay vì `<` khi pop

"Ấm hơn" là **lớn hơn hẳn**. Nếu pop cả khi bằng nhau thì ngày có cùng nhiệt độ được coi là ngày ấm hơn.

In [21]:
def wrong_le(t):
    n = len(t)
    ans = [0] * n
    st = []
    for i, x in enumerate(t):
        while st and t[st[-1]] <= x:             # SAI: phải là <
            j = st.pop()
            ans[j] = i - j
        st.append(i)
    return ans

print("sai :", wrong_le([60, 60, 70]), "| đúng:", Solution().dailyTemperatures([60, 60, 70]))
print("input nhỏ nhất làm sai:", find_mismatch(wrong_le))

sai : [1, 1, 0] | đúng: [2, 1, 0]
input nhỏ nhất làm sai: [31, 31]


**Vì sao sai:** ngày 0 (`60`) bị ngày 1 (`60`) "trả lời" với khoảng cách `1`, trong khi ngày ấm hơn đầu tiên thật sự là ngày 2 (`70`), cách `2`. **Code đúng:** `t[st[-1]] < x`.

### Lỗi 2: lưu **nhiệt độ** thay vì chỉ số vào ngăn xếp

Khi chỉ lưu nhiệt độ ta không biết `j` là ngày nào nên không tính được `i - j`; nhiều người đành dùng `ans[j]` sai chỗ.

In [22]:
def wrong_store_values(t):
    n = len(t)
    ans = [0] * n
    st = []                                      # SAI: chứa nhiệt độ
    for i, x in enumerate(t):
        while st and st[-1] < x:
            v = st.pop()
            ans[t.index(v)] = i - t.index(v)      # cố đoán chỉ số bằng t.index(v): sai khi có nhiệt độ lặp lại
        st.append(x)
    return ans

print("sai :", wrong_store_values([73, 74, 73, 75]), "| đúng:", Solution().dailyTemperatures([73, 74, 73, 75]))
print("input nhỏ nhất làm sai:", find_mismatch(wrong_store_values))

sai : [3, 2, 0, 0] | đúng: [1, 2, 1, 0]
input nhỏ nhất làm sai: [30, 30, 32]


**Vì sao sai:** `t.index(v)` luôn trả chỉ số **đầu tiên** có nhiệt độ `v`, nên các ngày lặp nhiệt độ bị gán đáp án sai/chồng chéo (và còn chậm, `O(n)` mỗi lần). **Code đúng:** đẩy **chỉ số** `i` vào ngăn xếp và tra nhiệt độ bằng `t[st[-1]]`.

### Lỗi 3: khởi tạo đáp án khác `0`

Đề yêu cầu `0` khi không có ngày ấm hơn. Nếu khởi tạo `ans = [-1] * n` (hay `None`) và quên gán lại cho các ngày còn trong ngăn xếp, đáp án sai.

In [23]:
def wrong_init(t):
    n = len(t)
    ans = [-1] * n                               # SAI: các ngày không có ngày ấm hơn phải là 0
    st = []
    for i, x in enumerate(t):
        while st and t[st[-1]] < x:
            j = st.pop()
            ans[j] = i - j
        st.append(i)
    return ans

print("sai :", wrong_init([73, 74, 75, 71]), "| đúng:", Solution().dailyTemperatures([73, 74, 75, 71]))
print("input nhỏ nhất làm sai:", find_mismatch(wrong_init))

sai : [1, 1, -1, -1] | đúng: [1, 1, 0, 0]
input nhỏ nhất làm sai: [34]


**Vì sao sai:** các chỉ số còn lại trong ngăn xếp khi hết vòng lặp không bao giờ được gán, nên giữ `-1`. **Code đúng:** `ans = [0] * n`.

### Lỗi 4: quét tuyến tính cho mỗi ngày và tưởng đủ nhanh

Cách 1 đúng nhưng `O(n^2)`. Với mảng dài `10^5` có nhiều nhiệt độ bằng nhau, nó chạy vài phút. Ta so cùng một input (độ dài vừa phải) giữa Cách 1 và Cách 2.

In [24]:
n = 8000
t = [50] * n
t0 = time.perf_counter(); a = daily_brute(t); t1 = time.perf_counter() - t0
t0 = time.perf_counter(); b = daily_stack(t); t2 = time.perf_counter() - t0
assert a == b
print(f"n={n}: quét tuyến tính {t1:.3f}s | ngăn xếp {t2:.4f}s | chậm hơn khoảng {t1 / t2:.0f} lần")
print("ước lượng cho n = 10^5 (tăng theo n^2):", round(t1 * (100000 / n) ** 2 / 60, 1), "phút")

n=8000: quét tuyến tính 1.073s | ngăn xếp 0.0007s | chậm hơn khoảng 1441 lần
ước lượng cho n = 10^5 (tăng theo n^2): 2.8 phút


**Vì sao chậm:** mỗi ngày quét hết phần còn lại, tổng `n(n-1)/2` so sánh. **Code đúng:** dùng ngăn xếp đơn điệu (hoặc cách nhảy) để mỗi ngày chỉ bị xử lý `O(1)` khấu hao.

## 14. Mẹo & nhận dạng pattern

- Bài hỏi "phần tử **đầu tiên lớn hơn** (hoặc nhỏ hơn) nằm **bên phải/bên trái**" → **ngăn xếp đơn điệu**.
- Nhớ ba quyết định: **lưu chỉ số**, ngăn xếp **không tăng** (tìm lớn hơn) hay **không giảm** (tìm nhỏ hơn), và dấu so sánh **chặt hay không** (bằng nhau có tính không).
- Mẫu mã: `for i, x in enumerate(a): while st and a[st[-1]] < x: j = st.pop(); ...; st.append(i)`.
- Mỗi phần tử push/pop một lần ⇒ tổng `O(n)` dù có `while` lồng `for`.
- Nếu đề cho khoảng giá trị rất nhỏ (như `30..100`) có thể đánh chỉ mục theo giá trị (Cách 4).
- Cách nhảy (Cách 3) tiết kiệm bộ nhớ nhưng khó chứng minh; ngăn xếp dễ tổng quát hơn (vòng tròn, trái/phải, cả "span").
- Test đừng quên: nhiệt độ bằng nhau, tăng chặt, giảm, một phần tử.

## 15. Biến thể

### 15.1. Số ngày **đã qua** kể từ lần ấm hơn gần nhất (nhìn về bên trái)

Đối xứng: duyệt từ phải sang trái, hoặc dùng ngăn xếp nhưng duyệt trái sang phải và trả lời cho chính ngày đang xét bằng đỉnh còn lớn hơn nó. Ở đây cho mỗi `i`: khoảng cách tới ngày **gần nhất bên trái** có nhiệt độ cao hơn hẳn, `0` nếu không có.

In [25]:
def prev_warmer(t):
    ans = [0] * len(t)
    st = []                                   # chứa chỉ số, nhiệt độ giảm chặt từ đáy lên đỉnh
    for i, x in enumerate(t):
        while st and t[st[-1]] <= x:          # bên trái: loại các ngày KHÔNG cao hơn x
            st.pop()
        if st:
            ans[i] = i - st[-1]
        st.append(i)
    return ans

def prev_warmer_brute(t):
    ans = [0] * len(t)
    for i in range(len(t)):
        for j in range(i - 1, -1, -1):
            if t[j] > t[i]:
                ans[i] = i - j
                break
    return ans

rng = random.Random(151)
for _ in range(300):
    t = [rng.randint(30, 40) for _ in range(rng.randint(1, 30))]
    assert prev_warmer(t) == prev_warmer_brute(t)
print(prev_warmer([73, 74, 75, 71, 69, 72, 76, 73]))

[0, 0, 0, 1, 1, 3, 0, 1]


Chú ý dấu: lần này ta pop với `<=` vì ta chỉ giữ lại các ngày **cao hơn hẳn** `x`. Cùng một họ bài nhưng dấu so sánh đổi theo ý nghĩa — đó là lý do phải luôn nghĩ về "bằng nhau có tính không".

### 15.2. Trả về **nhiệt độ** của ngày ấm hơn đầu tiên (Next Greater Element)

Thay vì khoảng cách, ghi giá trị `t[i]` của ngày trả lời (hoặc `-1` nếu không có). Đây là LeetCode 496/503.

In [26]:
def next_greater_values(t):
    ans = [-1] * len(t)
    st = []
    for i, x in enumerate(t):
        while st and t[st[-1]] < x:
            ans[st.pop()] = x
        st.append(i)
    return ans

print(next_greater_values([73, 74, 75, 71, 69, 72, 76, 73]))

[74, 75, 76, 72, 72, 76, -1, -1]


### 15.3. Mảng **vòng tròn** (circular)

Nếu sau ngày cuối lại là ngày đầu (vòng tròn), ta duyệt `2n` lần với chỉ số `i % n`. Bài tập 5 ở mục 17 yêu cầu bạn tự làm biến thể này cho khoảng cách.

### 15.4. Bài liên quan

- LeetCode 496 / 503 Next Greater Element I / II.
- LeetCode 901 Online Stock Span.
- LeetCode 84 Largest Rectangle in Histogram, 85 Maximal Rectangle.
- LeetCode 42 Trapping Rain Water (có cách ngăn xếp).
- LeetCode 316 Remove Duplicate Letters, 402 Remove K Digits (ngăn xếp đơn điệu trên chuỗi).

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1 Quét sang phải | mỗi ngày tự tìm | `O(n^2)` | `O(1)` | chỉ để hiểu đề / oracle |
| 2 Ngăn xếp đơn điệu | ngày mới trả lời cho các ngày chờ | `O(n)` | `O(n)` | **cách nộp**, chuẩn mực, tổng quát |
| 3 Nhảy theo đáp án | từ phải sang trái, nhảy `j += ans[j]` | `O(n)` khấu hao | `O(1)` | cần bộ nhớ phụ nhỏ nhất |
| 4 Bảng theo nhiệt độ | `nxt[v]` rồi lấy min các `v` lớn hơn | `O(n · V)` | `O(V)` | khi khoảng giá trị nhỏ và cố định |

## 17. Bài tập vận dụng

Làm lần lượt từ dễ đến khó. Gợi ý ở mục 18, đáp án ở mục 19.

### Bài 1 (đoán output)

Với `trace_stack` ở mục 8, đoán trạng thái ngăn xếp (chỉ số) **sau khi push** ở ngày cuối cùng, và đáp án cuối, cho `t = [50, 40, 60, 50, 55]`. Điền `ex1_answer = (stack_cuoi, answer)`. **Không** chạy trước khi đoán.

In [27]:
ex1_t = [50, 40, 60, 50, 55]
ex1_answer = None       # ví dụ ([0, 1], [0, 0, 0, 0, 0])  <- chỉ là mẫu định dạng

### Bài 2 (tính tay)

Tính tay `answer` cho `[89, 62, 70, 58, 47, 47, 46, 76, 100, 70]` (có nhiệt độ lặp `47, 47`). Điền `ex2_answer` là danh sách 10 số.

In [28]:
ex2_t = [89, 62, 70, 58, 47, 47, 46, 76, 100, 70]
ex2_answer = None

### Bài 3 (điền TODO)

Hoàn thành `ex3_daily`: hai chỗ `TODO` là điều kiện `while` (nhớ dấu so sánh!) và phần ghi đáp án. Cell này **không gọi** hàm.

In [29]:
def ex3_daily(t):
    n = len(t)
    ans = [0] * n
    st = []
    for i, x in enumerate(t):
        while False:                 # TODO 1: điều kiện pop (st không rỗng và nhiệt độ đỉnh ... x)
            j = st.pop()
            pass                     # TODO 2: ghi ans[j]
        st.append(i)
    return ans

### Bài 4 (tự viết lại)

Tự viết **không nhìn** hàm `ex4_rewrite(t)` bằng cách **nhảy** (Cách 3: từ phải sang trái, dùng `ans[j]`). Để `pass` nếu chưa làm.

In [30]:
def ex4_rewrite(t):
    pass

### Bài 5 (biến thể / mini challenge)

Viết `ex5_circular(t)`: giả sử các ngày xếp **thành vòng tròn** (sau ngày cuối là ngày đầu). Với mỗi `i`, trả về số ngày phải chờ để gặp ngày ấm hơn hẳn (đi vòng tối đa một vòng), `0` nếu không có. Ví dụ `[73, 74, 75, 71]` cho `[1, 1, 0, 1]`? Hãy tự suy ra (ngày 3 chờ: ngày 4 là ngày 0 với `73 > 71`, tức `1` ngày).

In [31]:
def ex5_circular(t):
    pass

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: mô phỏng ngăn xếp theo từng ngày.
- Mức 2: ngày 2 (`60`) pop `1` rồi `0`; ngày 3 (`50`) không pop; ngày 4 (`55`) pop `3`.
- Mức 3: sau ngày 4 ngăn xếp `[2, 4]`; `answer = [2, 1, 0, 1, 0]`.

**Bài 2**
- Mức 1: tìm ngày ấm hơn **hẳn** (lớn hơn) đầu tiên cho mỗi ngày.
- Mức 2: hai ngày `47, 47` đều phải chờ tới `76` ở ngày 7.
- Mức 3: ngày 4 chờ `3`, ngày 5 chờ `2`, ngày 6 chờ `1`; cả mảng `[8,1,5,4,3,2,1,1,0,0]`.

**Bài 3**
- Mức 1: pop khi đỉnh **thấp hơn hẳn** ngày mới.
- Mức 2: TODO 1: `st and t[st[-1]] < x`; TODO 2: `ans[j] = i - j`.
- Mức 3: đừng dùng `<=`.

**Bài 4**
- Mức 1: duyệt `i` từ `n - 2` về `0`, đặt `j = i + 1`.
- Mức 2: khi `t[j] <= t[i]`: nếu `ans[j] == 0` thì dừng (không có), ngược lại `j += ans[j]`.
- Mức 3: gán `ans[i] = j - i` chỉ khi tìm được `j < n`.

**Bài 5**
- Mức 1: duyệt `2n` ngày, chỉ số thật là `i % n`.
- Mức 2: ngăn xếp chỉ chứa chỉ số của vòng đầu (`i < n`); chỉ gán đáp án khi `ans[j]` còn `0` và chưa được trả lời.
- Mức 3: khi trả lời cho `j` bằng ngày `i` (có thể `i >= n`), khoảng cách là `i - j`.

## 19. Đáp án tham khảo

### Đáp án Bài 1

In [32]:
ex1_solution = ([2, 4], [2, 1, 0, 1, 0])
n1 = len(ex1_t)
st1, ans1 = [], [0] * n1
for i, x in enumerate(ex1_t):
    while st1 and ex1_t[st1[-1]] < x:
        j = st1.pop(); ans1[j] = i - j
    st1.append(i)
assert (st1, ans1) == ex1_solution
print("Bài 1 OK:", ex1_solution)

Bài 1 OK: ([2, 4], [2, 1, 0, 1, 0])


### Đáp án Bài 2

In [33]:
ex2_solution = [8, 1, 5, 4, 3, 2, 1, 1, 0, 0]
assert Solution().dailyTemperatures(ex2_t) == ex2_solution == daily_brute(ex2_t)
print("Bài 2 OK:", ex2_solution)

Bài 2 OK: [8, 1, 5, 4, 3, 2, 1, 1, 0, 0]


### Đáp án Bài 3

In [34]:
def ex3_solution(t):
    n = len(t)
    ans = [0] * n
    st = []
    for i, x in enumerate(t):
        while st and t[st[-1]] < x:
            j = st.pop()
            ans[j] = i - j
        st.append(i)
    return ans

rng = random.Random(3)
for _ in range(300):
    t = [rng.randint(30, 40) for _ in range(rng.randint(1, 40))]
    assert ex3_solution(t) == daily_brute(t)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [35]:
def ex4_solution(t):
    n = len(t)
    ans = [0] * n
    for i in range(n - 2, -1, -1):
        j = i + 1
        while t[j] <= t[i]:
            if ans[j] == 0:
                j = n
                break
            j += ans[j]
        if j < n:
            ans[i] = j - i
    return ans

rng = random.Random(4)
for _ in range(300):
    t = [rng.randint(30, 40) for _ in range(rng.randint(1, 40))]
    assert ex4_solution(t) == daily_brute(t)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [36]:
def ex5_solution(t):
    n = len(t)
    ans = [0] * n
    st = []
    for i in range(2 * n):
        x = t[i % n]
        while st and t[st[-1]] < x:
            j = st.pop()
            ans[j] = i - j
        if i < n:
            st.append(i)
    return ans

def ex5_oracle(t):
    n = len(t)
    ans = [0] * n
    for i in range(n):
        for step in range(1, n):
            if t[(i + step) % n] > t[i]:
                ans[i] = step
                break
    return ans

assert ex5_solution([73, 74, 75, 71]) == [1, 1, 0, 1]
rng = random.Random(5)
for _ in range(500):
    t = [rng.randint(30, 36) for _ in range(rng.randint(1, 30))]
    assert ex5_solution(t) == ex5_oracle(t), t
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template

```python
class Solution:
    def dailyTemperatures(self, temperatures: List[int]) -> List[int]:
        n = len(temperatures)
        ans = [0] * n                                    # mặc định 0: không có ngày ấm hơn
        st = []                                          # ngăn xếp CHỈ SỐ, nhiệt độ không tăng
        for i, x in enumerate(temperatures):
            while st and temperatures[st[-1]] < x:       # ấm hơn HẲN: dùng <
                j = st.pop()
                ans[j] = i - j                           # ngày i là ngày ấm hơn đầu tiên của j
            st.append(i)
        return ans
```

Câu thần chú: **"ngày mới trả lời cho các ngày chờ thấp hơn hẳn; lưu chỉ số; ans khởi tạo 0."**

### Checklist trước khi submit

- [ ] Ngăn xếp chứa **chỉ số**, không phải nhiệt độ.
- [ ] Điều kiện pop là `<` (chặt), không phải `<=`.
- [ ] `ans = [0] * n` (mặc định `0`).
- [ ] `ans[j] = i - j` (ngày trả lời trừ ngày được trả lời).
- [ ] Test: một phần tử, toàn bằng nhau, tăng chặt, giảm, có nhiệt độ lặp.
- [ ] Độ phức tạp `O(n)`; không quét tuyến tính lồng nhau cho mọi ngày.
- [ ] Skeleton đúng: `class Solution`, method `dailyTemperatures(self, temperatures: List[int]) -> List[int]`.